In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Import and Cleansing Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn import metrics

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df_daily = pd.read_csv('/kaggle/input/ihsgstockdata/daily/ANTM.csv')
df_daily

In [ ]:
df_daily.info()

In [ ]:
df_hourly = pd.read_csv('/kaggle/input/ihsgstockdata/hourly/ANTM.csv')
df_hourly

In [ ]:
df_hourly.info()

In [ ]:
df_minutes = pd.read_csv('/kaggle/input/ihsgstockdata/minutes/ANTM.csv')
df_minutes

In [ ]:
df_minutes.info()

In [ ]:
df_daily['timestamp'] = pd.to_datetime(df_daily['timestamp'])
df_daily.info()

In [ ]:
df_hourly['timestamp'] = pd.to_datetime(df_hourly['timestamp'])
df_hourly.info()

In [ ]:
df_minutes['timestamp'] = pd.to_datetime(df_minutes['timestamp'])
df_minutes.info()

In [ ]:
print(f'Describe of Each : \n Daily \n {df_daily.describe()}\n, Hourly \n{df_hourly.describe()}\n, Minutes \n{df_minutes.describe()}')

In [ ]:
print(df_daily.isnull().sum())
print(df_hourly.isnull().sum())
print(df_minutes.isnull().sum())

In [ ]:
features = ['open', 'high', 'low', 'close', 'volume']

# plt.subplots(figsize=(20,10))

for i, col in enumerate(features):
    plt.subplots(figsize=(20,10))
    plt.subplot(2,3,i+1)
    sns.distplot(df_daily[col])
    plt.show()

In [ ]:
import plotly.graph_objects as go
from datetime import datetime

fig = go.Figure(data=[go.Candlestick(x=df_daily['timestamp'],
                open=df_daily['open'],
                high=df_daily['high'],
                low=df_daily['low'],
                close=df_daily['close'])])

fig.show()

Disini saya akan menggunakan data pergerakan harga saham harian untuk memprediksi jangka menengah dengan begitu dataset hourly dan minutes akan saya hilangkan

In [ ]:
df = df_daily
df

In [ ]:
df_daily.describe()

In [ ]:
for i, col in enumerate(features):
    plt.subplots(figsize=(20,10))
    plt.subplot(2,3,i+1)
    sns.boxplot(df_daily[col])
    plt.show()

In [ ]:
df['timestamp']

# Feature Engineering

In [ ]:
# Create new columns
df['day'] = df['timestamp'].dt.day
df['month'] = df['timestamp'].dt.month
df['year'] = df['timestamp'].dt.year
df

In [ ]:
# menampilkan keterangan apakah harga tersebut berada di akhir kuartal atau tidak
# 1 = True
df['is_quarter_end'] = np.where(df['month']%3==0,1,0)
df.head()

In [ ]:
data_grouped = df.groupby('year').mean()
plt.subplots(figsize=(20,10))

for i, col in enumerate(['open', 'high', 'low', 'close']):
    plt.subplot(2,2,i+1)
    data_grouped[col].plot.bar()
plt.show()


In [ ]:
df.groupby('is_quarter_end').mean()

Dari tabel diatas dapat dilihat bahwa harga saham ANTM cenderung menurun setelah pengumuman laporan kuartal. Juga terdapat kondisi yang sama pada jumlah volume transaksi.

In [ ]:
df['open-close'] = df['open'] - df['close']
df['low-high'] = df['low'] - df['high']
df['target'] = np.where(df['close'].shift(-1) > df['close'], 1, 0)
df

In [ ]:
plt.pie(df['target'].value_counts().values,
        labels=[0, 1], autopct='%1.1f%%')
plt.show()

In [ ]:
plt.figure(figsize=(10, 10))

# menampilkan hubungan korelasi antar kolom untuk menentukan fitur machine learning
sns.heatmap(df.corr() > 0.9, annot=True, cbar=False)
plt.show()


# Data Pre-Processing

In [ ]:
features = df[['open-close', 'low-high', 'is_quarter_end']]
target = df['target']

scaler = StandardScaler()
features = scaler.fit_transform(features)

X_train, X_valid, Y_train, Y_valid = train_test_split(features, target, test_size=0.1, random_state=2022)
print(X_train.shape, X_valid.shape)

# Modelling

Disini kita melakukan training model machine learning (Logistic Regression, Support Vector Machine, XGBClassifier), dan kemudian akan dilihat kinerjanya berdasarkan data training dan validation sehingga akan terlihat model machine learning mana yang lebih baik untuk data tersebut.
Untuk evaluation metric, kita akan menggunakan kurva ROC-AUC dimana kurva ROC-AUC umumnya digunakan untuk mengukur akurasi prediksi.

In [ ]:
models = [LogisticRegression(), SVC(
kernel='poly', probability=True), XGBClassifier()]

for i in range(3):
    models[i].fit(X_train, Y_train)

print(f'{models[i]} : ')
print('Training Accuracy : ', metrics.roc_auc_score(Y_train, models[i].predict_proba(X_train)[:,1]))
print('Validation Accuracy : ', metrics.roc_auc_score(Y_valid, models[i].predict_proba(X_valid)[:,1]))
print()

In [ ]:
metrics.plot_confusion_matrix(models[0], X_valid, Y_valid)
plt.show()

Dari hasil Training Accuracy dan Validation Accuracy menunjukan bahwa model mampu mengklasifikasikan data dengan baik atau mampu mengklasifikasikan data validasi sebanyak 66% dari keseluruhan data validasi yang digunakan dengan benar. Ini bisa menjadi indikasi bahwa model tersebut memiliki kinerja yang baik dalam mengklasifikasikan data baru yang sejenis dengan data yang digunakan untuk melatih model. 